In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from scipy.stats import randint, uniform, loguniform
import warnings
warnings.filterwarnings('ignore')
import catboost as cb

In [ ]:
data = pd.read_excel('data/13.CatBoost best.xlsx')  

X = data.iloc[:, :-1]  
y = data.iloc[:, -1]   
feature_names = list(X.columns)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42                           ##0.3 0.7935； 0.4  0.6 
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.001, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.best_score = None
        self.counter = 0
        self.best_model = None
        self.best_params = None
        
    def __call__(self, score, model, params=None):
        if self.best_score is None:
            self.best_score = score
            self.best_model = model
            self.best_params = params
            if self.verbose:
        elif score < self.best_score - self.min_delta:
            improvement = self.best_score - score
            self.best_score = score
            self.counter = 0
            self.best_model = model
            self.best_params = params
            if self.verbose:
                print(f"MSE: {score:.6f})")
        else:
            self.counter += 1
            if self.verbose:
                print(f"   {self.counter}/{self.patience}, MSE: {score:.6f}")
            if self.counter >= self.patience:
                if self.verbose:
                    print(f"  🛑 best MSE: {self.best_score:.6f}")
                return True  
        return False 


def evaluate_model(model, X, y, set_name="val"):
    y_pred = model.predict(X)
    mse = mean_squared_error(y, y_pred)
    mae = mean_absolute_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    
    print(f"  {set_name} - MSE: {mse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}")
    return mse, mae, r2, y_pred

base_params = {
    'random_state': 42
}


base_model = cb.CatBoostRegressor(
    **base_params,
    verbose=False,
)

base_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    verbose=False
)

base_mse, base_mae, base_r2, _ = evaluate_model(base_model, X_val, y_val, "val")
base_test_mse, base_test_mae, base_test_r2, base_pred = evaluate_model(base_model, X_test, y_test, "test")

In [ ]:
# ============================================================================
# 3. Optuna
# ============================================================================

import optuna
from optuna.samplers import TPESampler

def objective(trial):
    iterations = trial.suggest_int('iterations', 100, 1000)
    depth = trial.suggest_int('depth', 4, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    l2_leaf_reg = trial.suggest_float('l2_leaf_reg', 1, 10)
    border_count = trial.suggest_int('border_count', 32, 255)
    random_strength = trial.suggest_float('random_strength', 0.1, 10)
    bagging_temperature = trial.suggest_float('bagging_temperature', 0, 1)
    model = cb.CatBoostRegressor(
        iterations=iterations,
        depth=depth,
        learning_rate=learning_rate,
        l2_leaf_reg=l2_leaf_reg,
        border_count=border_count,
        random_strength=random_strength,
        bagging_temperature=bagging_temperature,
        random_seed=42,
        verbose=False,
        early_stopping_rounds=50
    )
    

    model.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=False)
    
    y_val_pred = model.predict(X_val)
    mse = mean_squared_error(y_val, y_val_pred)
    
    return mse


study = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=42)
)


class OptunaEarlyStopping:
    def __init__(self, patience=15, min_delta=0.0001):
        self.patience = patience
        self.min_delta = min_delta
        self.best_value = None
        self.counter = 0
        
    def __call__(self, study, trial):
        if self.best_value is None:
            self.best_value = study.best_value
        elif study.best_value < self.best_value - self.min_delta:
            self.best_value = study.best_value
            self.counter = 0
            print(f"  Optuna find better! #{trial.number}, MSE: {study.best_value:.6f}")
        else:
            self.counter += 1
            
        if self.counter >= self.patience:
            print(f"  🛑 Optuna! best MSE: {self.best_value:.6f}")
            study.stop()
        else:
            if trial.number % 10 == 0:  
                print(f"  Optuna: #{trial.number}, MSE: {study.best_value:.6f}, num: {self.counter}/{self.patience}")


early_stopping_optuna = OptunaEarlyStopping(patience=15, min_delta=0.0001)


study.optimize(objective, n_trials=100, callbacks=[early_stopping_optuna])

print("\nOptuna finish!")
print("best_params:", study.best_params)
print("best val MSE:", study.best_value)


best_optuna_model = cb.CatBoostRegressor(
    **study.best_params,
    random_seed=42,
    verbose=False,
    early_stopping_rounds=50
)
best_optuna_model.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=False)

optuna_val_mse, optuna_val_mae, optuna_val_r2, _ = evaluate_model(best_optuna_model, X_val, y_val, "val")
optuna_train_mse, optuna_train_mae, optuna_train_r2, optuna_pred = evaluate_model(best_optuna_model, X_train, y_train, "test")

In [ ]:

params5 = best_optuna_model.get_params()

for key, value in params5.items():
    if value is not None:
        print(f"{key:25}: {value}")